# AXI test 5: does it hold at one million tokens? (real model, 3,600 steps)
Model: **Qwen2.5-7B-Instruct in 4-bit** (the same as tests 1 to 4). One long job: **3,600 requests in a row** about a small service graph (about 1.2 million tokens). At every step the model is shown only the exact current state and one request; the framework (the scope filter, `scope_m7`) decides what to keep. After every step the held state is compared with the right one. Predictions and the 13 reading rules were written and locked before this run in `docs/MILLION_TOKEN_TEST.md`.

* **Run 1** (about 70 to 110 minutes): the test itself.
* **Contrast** (about 20 minutes): the first 300 requests again with the three policies that do not have the scope filter (blind, gate, repair_m6).
* **Run 2** (optional, another 70 to 110 minutes): a second job with another seed, for "two million".

**Before you start:** Runtime menu -> *Change runtime type* -> **T4 GPU** -> Save. Run each cell **one at a time, top to bottom**, wait for the green tick before the next one. Cell 4 asks to connect Google Drive: allow it (that is where progress is saved).

**Keep this page open and your screen awake while Run 1 runs** (a phone that sleeps can disconnect Colab). If Colab stops you for any reason: nothing is lost. Reopen the notebook later and run **Cells 1 to 5 again, in order**, then the run cell again: it says `restored`, replays what was saved, checks it, and continues from the last saved step.

### Cell 1: check the GPU (it must say Tesla T4)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

### Cell 2: upload the code. A button 'Choose files' appears: pick **only** `axi-colab-million.zip`

In [ ]:
from google.colab import files
up = files.upload()
cands = [n for n in up if n.endswith(".zip") and "million" in n]
assert len(cands) == 1, f"I need exactly one file named like axi-colab-million.zip. I got: {list(up)}"
name = cands[0]
print("using", name)
!rm -rf /content/axi && mkdir -p /content/axi && unzip -q -o "{name}" -d /content/axi
!ls /content/axi/python

### Cell 3: install the 4-bit tools (about 1 minute)

In [ ]:
!pip -q install bitsandbytes accelerate

### Cell 4: connect Google Drive, load the model (downloads about 15 GB once; about 5 to 10 minutes)

In [ ]:
import sys, json, time, os, torch
from collections import deque
sys.path.insert(0, "/content/axi/python")
from colab.colab_llm_experiment import build_hf_llm
from colab.colab_persist import open_store
from axi.experiments.million import (run_million, verdict_million, print_million, print_verdict_million, gpu_probe,
                                     SEEDS, STEPS, CONTRAST_STEPS, CONTRAST_POLICIES, MAIN_POLICY, MODEL)

store = open_store("/content/out", "axi_million")
llm, model = build_hf_llm(MODEL, "cuda", load_4bit=True)
print("GPU:", torch.cuda.get_device_name(0), "| memory used (GB):", round(torch.cuda.memory_allocated() / 1e9, 1))

def make_progress(name):
    T0, first, recent, tokens = time.time(), [None], deque(maxlen=100), [0]
    def cb(t, n, rec):
        if first[0] is None: first[0] = t
        recent.append(rec); tokens[0] += (rec["prompt_tokens"] or 0) + rec["tokens"]
        if t % 100 == 0 or t == n:
            store.persist(name + ".jsonl")
            el = time.time() - T0; eta = el / (t - first[0] + 1) * (n - t)
            k = len(recent); gpu = rec.get("probe", {}).get("allocated_mb", "-")
            print(f"step {t}/{n} | {el/60:.0f} min so far, about {eta/60:.0f} min to go | tokens this session {tokens[0]:,} | last {k}: exact {100*sum(r['exact'] for r in recent)/k:.0f}%, "
                  f"mean distance {sum(r['distance'] for r in recent)/k:.2f}, {sum(r['seconds'] for r in recent)/k:.2f} s a step | GPU {gpu} MB")
    return cb

def save(res, name):
    path = store.path(name + ".json"); json.dump(res, open(path, "w")); store.persist(name + ".json"); return path

### Cell 5: quick check, 20 steps (about 1 to 2 minutes). If it fails, stop and send me the red error text

In [ ]:
smoke = run_million(llm, 99, 20, MAIN_POLICY, "/content/smoke.jsonl", meta=dict(model=MODEL, smoke=True), probe=gpu_probe, probe_every=10)
print_million(smoke["summary"], "smoke, 20 steps")
s = smoke["summary"]; per, sec = s["tokens"] / s["steps"], s["seconds"] / s["steps"]
print(f"\n{per:.0f} tokens a step and {sec:.2f} s a step: {STEPS} steps would be about {STEPS * per / 1e6:.2f} million tokens and {STEPS * sec / 60:.0f} minutes of model time")
for r in smoke["rows"][:6]: print(r["t"], r["kind"], "|", repr(r["text"])[:90])

### Cell 6: RUN 1. 3,600 steps, scope_m7 (about 70 to 110 minutes; a line is printed every 100 steps and progress is saved to Drive). If this is a new session it says `restored` and continues

In [ ]:
name = "axi_million_run1_seed%d" % SEEDS[0]
print("progress file:", store.restore(name + ".jsonl"))
res1 = run_million(llm, SEEDS[0], STEPS, MAIN_POLICY, store.path(name + ".jsonl"), meta=dict(model=MODEL), progress=make_progress(name), probe=gpu_probe)
print_million(res1["summary"], "RUN 1, scope_m7, seed %d" % SEEDS[0])
print_verdict_million(verdict_million(res1), "RUN 1 (lines 4, 12 and 13 need the next cells)")
files.download(save(res1, name))

### Cell 7: CONTRAST. The first 300 requests again with blind, gate and repair_m6 (about 20 minutes). Also prints the reading-rule lines

In [ ]:
contrast = {}
for p in CONTRAST_POLICIES:
    cname = "axi_million_contrast_%s_seed%d" % (p, SEEDS[0])
    print("progress file:", store.restore(cname + ".jsonl"))
    contrast[p] = run_million(llm, SEEDS[0], CONTRAST_STEPS, p, store.path(cname + ".jsonl"), meta=dict(model=MODEL), progress=make_progress(cname), probe=gpu_probe)
    print_million(contrast[p]["summary"], "CONTRAST, %s" % p)
print_verdict_million(verdict_million(res1, contrast), "RUN 1 and the contrast")
files.download(save(contrast, "axi_million_contrast_seed%d" % SEEDS[0]))

### Cell 8 (optional): RUN 2, the second seed, for 'two million' (another 70 to 110 minutes). Only if you have the time

In [ ]:
name2 = "axi_million_run2_seed%d" % SEEDS[1]
print("progress file:", store.restore(name2 + ".jsonl"))
res2 = run_million(llm, SEEDS[1], STEPS, MAIN_POLICY, store.path(name2 + ".jsonl"), meta=dict(model=MODEL), progress=make_progress(name2), probe=gpu_probe)
print_million(res2["summary"], "RUN 2, scope_m7, seed %d" % SEEDS[1])
print_verdict_million(verdict_million(res1, contrast, res2), "RUN 1, the contrast and RUN 2")
files.download(save(res2, name2))

### Done
Files downloaded: `axi_million_run1_seed20261021.json`, `axi_million_contrast_seed20261021.json` (and `axi_million_run2_seed20261022.json` if you ran Cell 8). If a download did not start, the same files are in your Google Drive, in the folder `axi_million`. Upload them to the chat, with a screenshot of the printed tables and the reading-rule lines. No key is needed or used.